In [1]:
import pandas as pd
import unicodedata
import re

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 3, Finished, Available, Finished)

In [2]:
def normalize_city(text):
    if pd.isna(text):  # keep NaN safe
        return text
    
    # remove accents/diacritics
    text = ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )
    
    # lowercase + strip spaces
    text = text.lower().strip()
    
    # collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text)
    
    # remove special characters like '-' or '/'and only keep apostrophe
    text = re.sub(r'[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+£]', ' ', text)
    
    return text

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 4, Finished, Available, Finished)

In [3]:
customers = pd.read_csv("/lakehouse/default/Files/olist_customers_dataset.csv",dtype={"customer_zip_code_prefix": str})
print(customers.head())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 5, Finished, Available, Finished)

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

  customer_zip_code_prefix          customer_city customer_state  
0                    14409                 franca             SP  
1                    09790  sao bernardo do campo             SP  
2                    01151              sao paulo             SP  
3                    08775        mogi das cruzes             SP  
4                    13056               campinas             SP  


In [4]:
# Convert all text columns to pandas string dtype
customers = customers.astype("string", copy=True, errors='raise')

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 6, Finished, Available, Finished)

In [5]:
assert all(customers.dtypes == "string")

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 7, Finished, Available, Finished)

In [6]:
# Inspect the shape of the dataframe
print(customers.shape)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 8, Finished, Available, Finished)

(99441, 5)


In [7]:
#Inspect the data types of each column and check for null values
print(customers.info())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 9, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: string(5)
memory usage: 3.8 MB
None


In [8]:
# Check for values that have anomalies in customer_city

# Look for special characters in the customer_city field 
char = r"[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+0£]"
mask = customers["customer_city"].astype(str).str.contains(char, regex=True)

# Filter the DataFrame
customers_with_special_chars = customers[mask]
print(customers_with_special_chars['customer_city'])
print("Total affected records:", len(customers_with_special_chars))

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 10, Finished, Available, Finished)

227          varre-sai
435         mogi-guacu
682         mogi-guacu
706         mogi-guacu
1362     pariquera-acu
             ...      
95726        xangri-la
96275       embu-guacu
96838       mogi-guacu
97044       mogi-guacu
99216       mogi-guacu
Name: customer_city, Length: 225, dtype: string
Total affected records: 225


In [9]:
# Run the normalize function on the customer_city column
customers['customer_city'] = customers['customer_city'].apply(normalize_city)
print(customers.head())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 11, Finished, Available, Finished)

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

  customer_zip_code_prefix          customer_city customer_state  
0                    14409                 franca             SP  
1                    09790  sao bernardo do campo             SP  
2                    01151              sao paulo             SP  
3                    08775        mogi das cruzes             SP  
4                    13056               campinas             SP  


In [10]:
# Run check for values that have anomalies in customer_city

# Look for special characters in the customer city field 
char = r"[-_/\\.,;:\"!?@#$%^&*()\[\]{}<>|~`=+0£]"
mask = customers["customer_city"].astype(str).str.contains(char, regex=True)

# Filter the DataFrame
customers_with_special_chars = customers[mask]
print(customers_with_special_chars['customer_city'])
print("Total affected records:", len(customers_with_special_chars))

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 12, Finished, Available, Finished)

Series([], Name: customer_city, dtype: object)
Total affected records: 0


In [11]:
# Read city_lookup reference table

city_lookup_by_zipcode = pd.read_parquet("abfss://datasquirrels@onelake.dfs.fabric.microsoft.com/SilverLakehouse.Lakehouse/Tables/dbo/city_lookup_by_zipcode")
print(city_lookup_by_zipcode)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 13, Finished, Available, Finished)

                  city state zip_start zip_end
0          00000-00999  None     00000   00999
1            São Paulo    SP     01000   05999
2               Osasco    SP     06000   06299
3          Carapicuíba    SP     06300   06399
4              Barueri    SP     06400   06499
...                ...   ...       ...     ...
22431          Charrua    RS     99960   99964
22432       Água Santa    RS     99965   99969
22433          Ciríaco    RS     99970   99979
22434  David Canabarro    RS     99980   99989
22435        Muliterno    RS     99990   99999

[22436 rows x 4 columns]


In [12]:
print(city_lookup_by_zipcode.info())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 14, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22436 entries, 0 to 22435
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   city       22436 non-null  object
 1   state      22418 non-null  object
 2   zip_start  22436 non-null  object
 3   zip_end    22436 non-null  object
dtypes: object(4)
memory usage: 701.3+ KB
None


In [13]:
# Run the normalize function on the city column
city_lookup_by_zipcode['city_customers_check'] = city_lookup_by_zipcode['city'].apply(normalize_city)
print(city_lookup_by_zipcode)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 15, Finished, Available, Finished)

                  city state zip_start zip_end city_customers_check
0          00000-00999  None     00000   00999          00000 00999
1            São Paulo    SP     01000   05999            sao paulo
2               Osasco    SP     06000   06299               osasco
3          Carapicuíba    SP     06300   06399          carapicuiba
4              Barueri    SP     06400   06499              barueri
...                ...   ...       ...     ...                  ...
22431          Charrua    RS     99960   99964              charrua
22432       Água Santa    RS     99965   99969           agua santa
22433          Ciríaco    RS     99970   99979              ciriaco
22434  David Canabarro    RS     99980   99989      david canabarro
22435        Muliterno    RS     99990   99999            muliterno

[22436 rows x 5 columns]


city_lookup_by_zipcode has columns called zip_start and zip end to signify the range of zipcode prefixes that a single city may have.

To use this range to lookup the customer_city using the customer_zip_code prefix, we need to convert the zip code columns in both tables to integers to be able to check within the range and print the matching city name from city_lookup_by_zipcode table.

In [14]:
# Keep original zip as string
customers["customer_zip_code_prefix_str"] = customers["customer_zip_code_prefix"]

# Cast to int for matching
customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].astype(int)

city_lookup_by_zipcode["zip_start"] = city_lookup_by_zipcode["zip_start"].astype(int)
city_lookup_by_zipcode["zip_end"] = city_lookup_by_zipcode["zip_end"].astype(int)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 16, Finished, Available, Finished)

In [15]:
def find_city(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["city_customers_check"]  # Return the first matching city
    return None

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 17, Finished, Available, Finished)

In [16]:
def find_state(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["state"]  # Return the first matching city
    return None

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 18, Finished, Available, Finished)

In [17]:
customers["matched_city"] = customers["customer_zip_code_prefix"].apply(find_city)
customers["matched_state"] = customers["customer_zip_code_prefix"].apply(find_state)
print(customers)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 19, Finished, Available, Finished)

                            customer_id                customer_unique_id  \
0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
...                                 ...                               ...   
99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   

       customer_zip_code_prefix          customer_city customer_state  \
0 

In [18]:
# Check the customers dataframes for rows with any null value
rows_with_nulls = customers[customers.isnull().any(axis=1)]
print(rows_with_nulls)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 20, Finished, Available, Finished)

Empty DataFrame
Columns: [customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, customer_zip_code_prefix_str, matched_city, matched_state]
Index: []


In [19]:
# Cross validate customer_city and matched_city
customers["validate_city_match"] = (customers["customer_city"] == customers["matched_city"])

# Cross validate customer_state and matched_state
customers["validate_state_match"] = (customers["customer_state"] == customers["matched_state"])

# Rows where either city or state validation failed
invalid_rows = customers[
    (~customers["validate_city_match"]) | (~customers["validate_state_match"])
]

print(invalid_rows)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 21, Finished, Available, Finished)

                            customer_id                customer_unique_id  \
54     8247b5583327ab8be19f96e1fb82f77b  d85547cd859833520b311b4458a14c1c   
142    d387341bbce5ab96e3647bb0e8d0b55a  91e6fd64694fa9a828270d442ba88e03   
215    e48f50d453ada5bceb41e239a2dc2065  9d85c39a38a86ed13978803d7560c612   
261    4308615296cf4ca17defd8b0d59287d8  a99021699fda65c2b84a3fb596a4c32b   
594    be4ff2f8c48c87270bf96c870b2f7a23  5b424d5287408acfeb03ad7fed722545   
...                                 ...                               ...   
98878  78a11bb1fa72f556996b9a5b9bcd0629  e7536f62a200b415edd9491ac12a17fa   
99047  1e44951cea9cac0d305ae89e6b2247ee  bd7287fdad74d4d53c8d2b6e1cfd644d   
99072  18e56af97c2f24afcfaf7aa97ad2b969  d9ff7e37d7bf448abb783a6e56462d93   
99135  2ed1b5c01561bcf39cc2f1566a8ac9f1  ed05a1bbf9f8816af23fcc68084bb87d   
99224  1f7d089d663f7a5be55536aa0a49bbbc  a6d1eaa6d3f0a8f5440b3ba9d6331890   

       customer_zip_code_prefix     customer_city customer_state  \
54     

There are 680 rows where either the city or the state does not match. We'll make the assumption that the zip code prefix provided in the customers table to be true and update the customer_city and customer_state using the city_lookup_by_zipcode table to ensure consistency in the sellers data. 

We will take the original city name and state name from the city_look_up_by_zipcode for consistency with Brazil's naming convention.

In [20]:
def find_city_lookup(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["city"]  # Return the first matching city
    return None

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 22, Finished, Available, Finished)

In [21]:
def find_state_lookup(zipcode):
    matches = city_lookup_by_zipcode[
        (zipcode >= city_lookup_by_zipcode["zip_start"]) &
        (zipcode <= city_lookup_by_zipcode["zip_end"])
    ]
    if not matches.empty:
        return matches.iloc[0]["state"]  # Return the first matching state
    return None

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 23, Finished, Available, Finished)

In [22]:
customers["lookup_city"] = customers["customer_zip_code_prefix"].apply(find_city_lookup)
customers["lookup_state"] = customers["customer_zip_code_prefix"].apply(find_state_lookup)
print(customers)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 24, Finished, Available, Finished)

                            customer_id                customer_unique_id  \
0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
...                                 ...                               ...   
99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   

       customer_zip_code_prefix          customer_city customer_state  \
0 

In [23]:
#Drop the unnecessary columns

drop_columns = ["customer_zip_code_prefix", "customer_city", "customer_state", "matched_city", "matched_state", "validate_city_match", "validate_state_match"]
customers_cleaned = customers.drop(drop_columns, axis=1)
print(customers_cleaned.head())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 25, Finished, Available, Finished)

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

  customer_zip_code_prefix_str            lookup_city lookup_state  
0                        14409                 Franca           SP  
1                        09790  São Bernardo do Campo           SP  
2                        01151              São Paulo           SP  
3                        08775        Mogi das Cruzes           SP  
4                        13056               Campinas           SP  


In [24]:
# Replace column names
col_names = ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]
customers_cleaned.columns = col_names
print(customers_cleaned.head())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 26, Finished, Available, Finished)

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

  customer_zip_code_prefix          customer_city customer_state  
0                    14409                 Franca             SP  
1                    09790  São Bernardo do Campo             SP  
2                    01151              São Paulo             SP  
3                    08775        Mogi das Cruzes             SP  
4                    13056               Campinas             SP  


In [25]:
# Convert all text columns to pandas string dtype
customers_cleaned = customers_cleaned.astype("string", copy=True, errors='raise')

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 27, Finished, Available, Finished)

In [26]:
assert all(customers_cleaned.dtypes == "string")

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 28, Finished, Available, Finished)

In [27]:
# Check final customers_cleaned dataframe that all rows and columns from original dataframe is consistent after clean up
print(customers_cleaned.info())

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 29, Finished, Available, Finished)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  string
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: string(5)
memory usage: 3.8 MB
None


In [28]:
# Write the table to the silver lakehouse as a delta table
# Convert pandas to Spark
spark_customers = spark.createDataFrame(customers_cleaned)

# Save as a Delta table in Silver Lakehouse
silver_path = "SilverLakehouse.dbo.olist_customers_cleaned"
spark_customers.write.format("delta").mode("overwrite").saveAsTable(silver_path)

StatementMeta(, ef7126a1-4d70-4969-a241-a0231c70d7dd, 30, Finished, Available, Finished)